# Cell Translator — Morphological perturbation: Phase 1 interpretation

**Purpose:** post-training analysis of existing, held-out drug-effect results. **No AE/CycleGAN retraining.** This notebook reads the `evaluation/*_feature_effects.csv` files generated by the QC rerun, validates their consistency, and creates publication-oriented diagnostics.

It distinguishes (A) developmental **perturbed iPSC → perturbed astrocyte** prediction, whose drug effect is calculated against developmentally translated DMSO astrocytes, from (B) **astrocyte DMSO → astrocyte drug** translation, whose predicted effect is compared with measured DMSO astrocytes.

**Interpretation boundaries:** Test-set features are not biological replicates. Family counts, feature-level rankings and across-feature summaries are descriptive, not inferential evidence of generalization to new donors, plates or drugs. No model selection or threshold tuning should be performed using final test data.

In [ ]:
import os
from pathlib import Path
import json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

# Update only if you changed the rerun folder in the training notebook.
RUN_ROOT = Path(os.environ.get('CELL_TRANSLATOR_RUN_DIR', str(Path.cwd() / 'outputs' / 'rerun_70_15_15_qc_v3'))).expanduser().resolve()
EVAL_DIR = RUN_ROOT / 'evaluation'
OUT = EVAL_DIR / 'phase1_interpretation'
OUT.mkdir(parents=True, exist_ok=True)
DRUGS = ['Latrunculin_B', 'GW3965', 'Clozapine']
MODELS = ['developmental', 'control_to_drug']
FAMILIES = ['AreaShape','Granularity','RadialDistribution','Texture','Intensity','Correlation','Other']
TOP_K = [20, 50, 100]
MIN_FAMILY_FEATURES_FOR_CORR = 10

assert EVAL_DIR.is_dir(), f'Missing evaluation directory: {EVAL_DIR}'
print('Reading from:', EVAL_DIR)
print('Writing to:', OUT)

## 1. Load and audit saved feature-level effects

Each source table must contain `feature`, `real_effect`, and `pred_effect`. These are **standardized drug-effect vectors**, computed in the rerun as drug-minus-DMSO changes divided by measured astrocyte pooled SD. We check feature identities and nonfinite values before analysis. No column-position matching is permitted.

In [ ]:
def family_of(name):
    for family in FAMILIES[:-1]:
        if family.lower() in str(name).lower():
            return family
    return 'Other'

def safe_corr(x, y, method='pearson'):
    a = np.asarray(x, dtype=float); b = np.asarray(y, dtype=float)
    good = np.isfinite(a) & np.isfinite(b)
    if good.sum() < 3 or np.std(a[good]) <= 1e-12 or np.std(b[good]) <= 1e-12:
        return np.nan
    fun = pearsonr if method == 'pearson' else spearmanr
    return float(fun(a[good], b[good]).statistic)

tabs = {}
audit = []
for drug in DRUGS:
    for model in MODELS:
        tag = f'{model}__{drug}'
        path = EVAL_DIR / f'{tag}_feature_effects.csv'
        if not path.is_file():
            raise FileNotFoundError(f'Missing {path} — rerun evaluation cell 32 first')
        df = pd.read_csv(path)
        needed = {'feature','real_effect','pred_effect'}
        assert needed.issubset(df), f'{tag}: missing {needed-set(df)}'
        assert df.feature.notna().all() and df.feature.is_unique, f'{tag}: bad or duplicate feature names'
        for col in ['real_effect','pred_effect']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        assert np.isfinite(df[['real_effect','pred_effect']].to_numpy()).all(), f'{tag}: nonfinite effects'
        df['family'] = df['feature'].map(family_of)
        df['abs_real'] = df.real_effect.abs()
        df['abs_pred'] = df.pred_effect.abs()
        df['error'] = df.pred_effect - df.real_effect
        df['abs_error'] = df.error.abs()
        df['sign_match'] = np.sign(df.real_effect) == np.sign(df.pred_effect)
        tabs[tag] = df.set_index('feature', drop=False)
        audit.append({'comparison':tag,'n_features':len(df),'n_missing_or_nonfinite':0,
                      'n_families':df.family.nunique(),
                      'n_real_effects_abs_lt_0.05':int((df.abs_real < 0.05).sum())})

for drug in DRUGS:
    a=tabs[f'developmental__{drug}']; b=tabs[f'control_to_drug__{drug}']
    assert set(a.index)==set(b.index), f'{drug}: strategies use different feature panels; align first'
    realdiff=np.abs(a.loc[b.index,'real_effect']-b.real_effect)
    assert realdiff.max()<1e-8, f'{drug}: ground-truth effects differ across strategies'

audit_df=pd.DataFrame(audit)
audit_df.to_csv(OUT/'input_feature_table_audit.csv',index=False)
display(audit_df)

## 2. Effect calibration: does the model recover magnitudes as well as directions?

The no-effect prediction is a vector of zeros. Compare model MAE against this baseline, report **signed bias**, the slope of predicted on measured (through the origin), the magnitude ratio, and sign concordance specifically among the strongest measured effects. Pearson correlation does not diagnose amplitude errors.

In [ ]:
calib=[]
for drug in DRUGS:
    for model in MODELS:
        df=tabs[f'{model}__{drug}']
        x=df.real_effect.to_numpy(float); y=df.pred_effect.to_numpy(float)
        mae=np.mean(np.abs(y-x)); baseline=np.mean(np.abs(x))
        strong=np.abs(x)>=np.quantile(np.abs(x),0.90)
        slope=(x@y)/(x@x) if x@x>1e-12 else np.nan
        calib.append({'drug':drug,'model':model,'features':len(df),
            'pearson':safe_corr(x,y),'spearman':safe_corr(x,y,'spearman'),
            'MAE':mae,'no_effect_MAE':baseline,'MAE_improvement_pct':100*(baseline-mae)/baseline if baseline else np.nan,
            'signed_bias':np.mean(y-x),'slope_through_origin':slope,
            'mean_abs_effect_ratio':np.mean(np.abs(y))/np.mean(np.abs(x)) if np.mean(np.abs(x)) else np.nan,
            'sign_concordance':np.mean(np.sign(x)==np.sign(y)),
            'top10pct_sign_concordance':np.mean(np.sign(x[strong])==np.sign(y[strong]))})
calibration=pd.DataFrame(calib)
calibration.to_csv(OUT/'effect_calibration_and_noeffect_baseline.csv',index=False)
display(calibration.round(4))

fig, axes=plt.subplots(2,3,figsize=(15,9),sharex=False,sharey=False)
for i,drug in enumerate(DRUGS):
    for j,model in enumerate(MODELS):
        ax=axes[j,i]; df=tabs[f'{model}__{drug}']; x=df.real_effect; y=df.pred_effect
        ax.scatter(x,y,s=8,alpha=.35,rasterized=True)
        lim=max(np.quantile(np.abs(np.r_[x,y]),.995),.15)*1.08
        ax.plot([-lim,lim],[-lim,lim],ls='--',lw=1,label='Perfect recovery')
        ax.axhline(0,lw=.6);ax.axvline(0,lw=.6)
        ax.set(xlim=(-lim,lim),ylim=(-lim,lim),xlabel='Measured astrocyte drug effect',ylabel='Predicted astrocyte drug effect',title=f'{drug} — {model}')
        row=calibration[(calibration.drug==drug)&(calibration.model==model)].iloc[0]
        ax.text(.03,.97,f'r={row.pearson:.3f}; slope={row.slope_through_origin:.2f}\nMAE change={row.MAE_improvement_pct:+.1f}%',transform=ax.transAxes,va='top',fontsize=9)
fig.tight_layout();fig.savefig(OUT/'fig_effect_calibration.png',dpi=220,bbox_inches='tight');plt.show()

## 3. Morphology-family recovery, using both direction and magnitude

Family-level Pearson correlations can be noisy when only a few features belong to a family. The table reports feature counts and correlations only for families with at least `MIN_FAMILY_FEATURES_FOR_CORR`. Also evaluate mean absolute error, model/no-effect baseline improvement, mean signed effect, and overlap of directional responses. An average signed family effect can hide opposing individual features; inspect both signed and absolute effects.

In [ ]:
family_rows=[]
for drug in DRUGS:
    for model in MODELS:
        df=tabs[f'{model}__{drug}']
        for fam,g in df.groupby('family'):
            x=g.real_effect.to_numpy(); y=g.pred_effect.to_numpy()
            n=len(g); baseline=np.mean(np.abs(x)); mae=np.mean(np.abs(x-y))
            family_rows.append({'drug':drug,'model':model,'family':fam,'n_features':n,
                'pearson_r':safe_corr(x,y) if n>=MIN_FAMILY_FEATURES_FOR_CORR else np.nan,
                'spearman_r':safe_corr(x,y,'spearman') if n>=MIN_FAMILY_FEATURES_FOR_CORR else np.nan,
                'sign_concordance':np.mean(np.sign(x)==np.sign(y)),
                'mean_real_effect':np.mean(x),'mean_pred_effect':np.mean(y),
                'mean_abs_real_effect':np.mean(np.abs(x)),'mean_abs_pred_effect':np.mean(np.abs(y)),
                'MAE':mae,'no_effect_MAE':baseline,
                'MAE_improvement_pct':100*(baseline-mae)/baseline if baseline else np.nan})
family=pd.DataFrame(family_rows)
family.to_csv(OUT/'morphology_family_interpretation.csv',index=False)
display(family.sort_values(['drug','model','family']).round(3))

fig, axes=plt.subplots(2,3,figsize=(16,8))
for i,drug in enumerate(DRUGS):
    for j,model in enumerate(MODELS):
        ax=axes[j,i]
        x=family[(family.drug==drug)&(family.model==model)].set_index('family').reindex(FAMILIES)
        y=np.arange(len(FAMILIES));h=.35
        ax.barh(y-h/2,x.mean_abs_real_effect,h,label='Measured |effect|')
        ax.barh(y+h/2,x.mean_abs_pred_effect,h,label='Predicted |effect|')
        ax.set(yticks=y,yticklabels=FAMILIES,xlabel='Mean absolute standardized effect',title=f'{drug} — {model}')
        ax.invert_yaxis()
axes[0,0].legend(loc='upper right',fontsize=8)
fig.tight_layout();fig.savefig(OUT/'fig_family_effect_magnitude.png',dpi=220,bbox_inches='tight');plt.show()

# Separate heatmap for family-wise Pearson correlations (descriptive, no tests).
fig,axes=plt.subplots(1,2,figsize=(13,5),sharey=True)
for ax,model in zip(axes,MODELS):
    pivot=family[family.model==model].pivot(index='family',columns='drug',values='pearson_r').reindex(FAMILIES)[DRUGS]
    im=ax.imshow(pivot.to_numpy(float),vmin=-1,vmax=1,cmap='coolwarm',aspect='auto')
    ax.set(xticks=np.arange(3),xticklabels=DRUGS,yticks=np.arange(len(FAMILIES)),yticklabels=FAMILIES,title=model)
    for (r,c),value in np.ndenumerate(pivot.to_numpy()):
        ax.text(c,r,'NA' if pd.isna(value) else f'{value:.2f}',ha='center',va='center',fontsize=9)
fig.colorbar(im,ax=list(axes),shrink=.75,label='Feature-level effect Pearson r')
fig.subplots_adjust(left=.15,right=.87,bottom=.15,wspace=.25)
fig.savefig(OUT/'fig_family_effect_correlation.png',dpi=220,bbox_inches='tight');plt.show()

## 4. Which individual morphology features are recovered, missed, or predicted with the wrong sign?

Use the developmental translator as the main biological analysis; export parallel tables for control-to-drug. A "missed" feature has a large measured effect and a small predicted effect; a "sign reversal" has a large measured effect and the opposite predicted sign. These are **descriptive rankings**, not independently validated drug targets.

In [ ]:
feature_rows=[]
for drug in DRUGS:
    for model in MODELS:
        df=tabs[f'{model}__{drug}'].copy()
        q90=df.abs_real.quantile(.90)
        strong=df[df.abs_real>=q90].copy()
        strong['category']=np.select([
            np.sign(strong.real_effect)!=np.sign(strong.pred_effect),
            strong.abs_pred < strong.abs_real*.5,
            strong.abs_error <= strong.abs_real*.5],
            ['opposite_direction','underpredicted_magnitude','recovered_direction_and_magnitude'],
            default='same_direction_other_magnitude')
        strong=strong.sort_values('abs_real',ascending=False)
        strong.to_csv(OUT/f'{model}__{drug}_top10pct_measured_effect_features.csv',index=False)
        summary=strong.groupby(['family','category']).size().rename('n').reset_index()
        summary.insert(0,'drug',drug);summary.insert(1,'model',model)
        feature_rows.append(summary)
        display(pd.DataFrame({'drug':[drug],'model':[model],
          'strongest_measured_features':[len(strong)],
          'opposite_sign':[int((strong.category=='opposite_direction').sum())],
          'underpredicted_magnitude':[int((strong.category=='underpredicted_magnitude').sum())]}))
        # Top 20 measured features, with corresponding predictions
        strong.head(20)[['feature','family','real_effect','pred_effect','error','category']].to_csv(
            OUT/f'{model}__{drug}_top20_measured_effects.csv',index=False)
feature_categories=pd.concat(feature_rows,ignore_index=True)
feature_categories.to_csv(OUT/'feature_recovery_categories_by_family.csv',index=False)

# Comparative feature-level MAE: use strictly paired features in each drug.
model_comparison=[]
for drug in DRUGS:
    a=tabs[f'developmental__{drug}'].sort_index()
    b=tabs[f'control_to_drug__{drug}'].sort_index()
    assert a.index.equals(b.index)
    model_comparison.append({'drug':drug,'n_features':len(a),
        'developmental_lower_abs_error_fraction':float(np.mean(a.abs_error < b.abs_error)),
        'equal_error_fraction':float(np.mean(np.isclose(a.abs_error,b.abs_error))),
        'developmental_MAE':a.abs_error.mean(),
        'control_to_drug_MAE':b.abs_error.mean()})
model_comparison=pd.DataFrame(model_comparison)
model_comparison.to_csv(OUT/'strategy_comparison_paired_features.csv',index=False)
display(model_comparison.round(4))

## 5. Strongest-feature rank recovery and cross-drug specificity

Interpret absolute-effect rank overlap separately from global effect correlation. Evaluate drug specificity **on one common feature panel across all three drugs**. A diagonal maximum is promising but only three drugs are available, so formal generalization claims are not justified.

In [ ]:
rank_rows=[]
for drug in DRUGS:
    for model in MODELS:
        df=tabs[f'{model}__{drug}']
        for k0 in TOP_K:
            k=min(k0,len(df)); a=set(df.abs_real.nlargest(k).index); b=set(df.abs_pred.nlargest(k).index)
            rank_rows.append({'drug':drug,'model':model,'top_k':k,'overlap':len(a&b),
                              'recall_at_k':len(a&b)/k,'jaccard':len(a&b)/len(a|b)})
rank=pd.DataFrame(rank_rows);rank.to_csv(OUT/'top_feature_rank_recovery.csv',index=False)
display(rank.round(3))

spec_rows=[]
for model in MODELS:
    t={d:tabs[f'{model}__{d}'] for d in DRUGS}
    common=sorted(set.intersection(*(set(z.index) for z in t.values())))
    assert len(common)>=3
    mat=pd.DataFrame(index=DRUGS,columns=DRUGS,dtype=float)
    for predicted in DRUGS:
        for real in DRUGS:
            mat.loc[predicted,real]=safe_corr(t[predicted].loc[common,'pred_effect'],t[real].loc[common,'real_effect'])
    mat.to_csv(OUT/f'{model}_cross_drug_specificity_matched_features.csv')
    for d in DRUGS:
        own=mat.loc[d,d]; others=mat.loc[d].drop(d)
        spec_rows.append({'model':model,'drug':d,'n_common_features':len(common),
                          'own_drug_r':own,'largest_other_drug_r':others.max(),
                          'specificity_margin':own-others.max(),'correct_top_match':bool(mat.loc[d].idxmax()==d)})
    display(mat.round(3))
spec=pd.DataFrame(spec_rows);spec.to_csv(OUT/'cross_drug_specificity_margins.csv',index=False)
display(spec.round(3))

## 6. Provenance and independence audit (no automatic pass)

Saved test indices only establish that cells were placed in a test partition; they **cannot establish** that donors or replicate groups were held out across conditions or that test data were never used in model selection. This cell checks what is verifiable from disk and reports remaining limitations. If group labels exist, examine shared biological group assignments across all conditions before claiming independent donor-level prediction.

In [ ]:
AE_DIR=RUN_ROOT/'autoencoders'
conditions=[f'{t}_{d}' for t in ['iPSC','Astrocytes'] for d in ['DMSO']+DRUGS]
rows=[]; group_assign={}
for condition in conditions:
    p=AE_DIR/f'{condition}_split_assignments.csv'
    if not p.exists():
        rows.append({'condition':condition,'split_file_exists':False});continue
    s=pd.read_csv(p)
    counts=s['split'].value_counts().to_dict()
    row={'condition':condition,'split_file_exists':True,'n_train':counts.get('train',0),
         'n_val':counts.get('val',0),'n_test':counts.get('test',0),
         'n_total':len(s),'row_id_unique':s.row_id.is_unique}
    if 'group' in s and s['group'].notna().any() and s['group'].astype(str).str.strip().ne('').any():
        z=s[['group','split']].dropna().copy()
        z=z[z.group.astype(str).str.strip().ne('')]
        cross=z.groupby('group')['split'].nunique()
        row['n_groups_in_multiple_splits_within_condition']=int((cross>1).sum())
        group_assign[condition]=z.groupby('group')['split'].first().to_dict()
    else:
        row['n_groups_in_multiple_splits_within_condition']=np.nan
    rows.append(row)
provenance=pd.DataFrame(rows);provenance.to_csv(OUT/'split_provenance_audit.csv',index=False)
display(provenance)
if group_assign:
    conflicts=[]
    combined={}
    for condition,group_to_split in group_assign.items():
        for group,split in group_to_split.items():
            combined.setdefault(group,{}).setdefault(split,[]).append(condition)
    for group,by_split in combined.items():
        if len(by_split)>1:
            conflicts.append({'group':group,'split_assignments':str(by_split)})
    pd.DataFrame(conflicts,columns=['group','split_assignments']).to_csv(OUT/'cross_condition_group_split_conflicts.csv',index=False)
    print('Cross-condition group conflicts:',len(conflicts))
else:
    print('No usable group metadata found; donor/replicate-level independence cannot be verified.')
print('Checkpoint-selection and training-only latent fitting must additionally be verified from the training code/logs.')

## 7. What the results can and cannot conclude

- **Effect recovery**: report Pearson/Spearman, sign concordance, magnitude error and baseline comparison together. A model can have strong positive correlation but worse MAE than zero-effect prediction.
- **Biological interpretation**: discuss feature families with measured/predicted direction and amplitude, rather than assigning biological mechanisms from feature names alone.
- **Specificity**: correct matching across the three tested drugs shows discrimination *within these drugs*; it is not an unseen-drug evaluation.
- **Generalization**: when only cell-level random splits are used, results concern held-out cells from the sampled conditions, **not** unseen donors, lines, or perturbations.
- **Feature statistics**: highly correlated CellProfiler features make feature counts and rankings non-independent. Do not treat them as biological replicate p-values.

**Recommended next input:** share `morphology_family_interpretation.csv`, `effect_calibration_and_noeffect_baseline.csv`, and the top-feature CSVs or figures. We can then draft the Results and Discussion with only directly supported biological claims.